# 05. 추론·출력 형식 제어 — Tip 13 / 16 / 17 / 18 실전 검증

이 노트북은 프롬프트 작성 팁 4개를 `gpt-5-nano` 모델로 **직접 호출해** 검증합니다.
각 팁마다 (A) 개선 전 / (B) 개선 후 프롬프트를 나란히 돌려보고, `compare(...)` · `ask_meta(...)` · `keyword_hits(...)` 함수로 두 결과의 차이를 눈으로 확인합니다.

- **Tip 13 — 지시대명사 대신 풀네임 반복**: 모델은 "그것 / 위 자료 / 앞의 것"이 무엇을 가리키는지 잘 놓친다. 여러 단계를 거슬러 올라가야 하는 질문에서 (A) 지시대명사 vs (B) 풀네임 반복의 정답률을 비교한다.
- **Tip 16 — 추론량 줄이기(파라미터 + 자연어)**: `reasoning_effort="low"` 파라미터(내부 추론을 얼마나 돌릴지 정하는 설정) 하나만으로는 부족할 수 있다. 자연어 지시로도 "추론을 최소화하라"를 함께 적어준다. (A) 기본 vs (B) `reasoning_effort="minimal"` + 시스템 지시를 `reasoning_tokens` / `latency`로 실측 비교한다.
- **Tip 17 — verbosity(출력 분량) 조절**: 답이 너무 짧을 때는 `verbosity="high"`(출력을 얼마나 길게 할지 정하는 파라미터) + "자세히" 지시로 분량을 늘릴 수 있다. (A) low vs (B) high 의 길이 / 토큰을 비교한다.
- **Tip 18 — 리스트 대신 산문으로**: 최신 모델은 설명을 요청하면 1,2,3 번호나 하이픈 리스트로 답하는 경향이 있다. 줄글(산문)을 원하면 "리스트·하이픈 금지, 줄글로"라고 명시한다. (A) 기본 vs (B) 산문 지시의 형식 차이를 비교한다.

> ⚠️ 아래 셀들은 OpenAI API를 실제로 호출합니다(크레딧이 소모됩니다). 위에서 아래로 순서대로 Run 하세요.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## Tip 13 — 지시대명사 대신 풀네임 반복

**요지**: "그것 / 위 자료 / 앞의 것 / 그 회사" 같은 지시적 표현은, 가리킬 수 있는 대상이 여럿이고 여러 단계를 거슬러 올라가야 할 때 모델이 엉뚱한 대상으로 연결하기 쉽다. 그래서 실무에서는 **매번 고유명(풀네임)을 다시 적어주는 편이 안전하다.**

**무엇을 어떻게 검증하나**: 3단계로 이어지는 의존 관계(아폴로 → 헤르메스 → 제우스)를 준 뒤, 마지막 프로젝트가 무엇을 만드는지 묻는 **여러 단계를 거슬러 올라가는 질문**을 던진다.
- (A) 질문을 "그것이 의존하는… 다시 그것이 의존하는…" 처럼 **지시대명사만으로** 구성
- (B) 매 단계에서 **프로젝트 풀네임을 다시 적도록** 구성

정답은 `추천 시스템`(제우스). `keyword_hits`로 답에 "추천"이 들어갔는지 자동 채점해 비교한다. (소형 모델이라 실행할 때마다 결과가 흔들릴 수 있으니 여러 번 Run 해보라.)

In [ ]:
# === Tip 13: 지시대명사(A) vs 풀네임 반복(B) — 다단계 지칭 정확도 ===
# 의존 체인: 아폴로(결제) -> 헤르메스(알림) -> 제우스(추천). 3-hop 을 따라가야 정답.

facts = (
    "세 개의 프로젝트가 있다.\n"
    "- 아폴로 프로젝트: 2021년 시작, 결제 시스템을 만든다.\n"
    "- 헤르메스 프로젝트: 2022년 시작, 알림 시스템을 만든다.\n"
    "- 제우스 프로젝트: 2023년 시작, 추천 시스템을 만든다.\n"
    "의존 관계: 아폴로는 헤르메스에 의존한다. 헤르메스는 제우스에 의존한다."
)

# (A) 지시대명사로만 지칭 — "그것 / 그 프로젝트"
prompt_A = facts + "\n\n" + (
    "질문: 결제 시스템을 만드는 그 프로젝트, 그것이 의존하는 프로젝트, "
    "다시 그것이 의존하는 프로젝트 — 그 마지막 프로젝트는 무엇을 만드는가? "
    "결론을 한 문장으로만 답하라."
)

# (B) 매 단계 풀네임(아폴로/헤르메스/제우스)을 다시 적도록 강제
prompt_B = facts + "\n\n" + (
    "질문: 아폴로 프로젝트가 의존하는 프로젝트, 그 프로젝트가 다시 의존하는 프로젝트를 "
    "차례로 따라가, 마지막 프로젝트가 무엇을 만드는지 답하라. "
    "추적하는 매 단계에서 프로젝트의 풀네임(아폴로/헤르메스/제우스)을 반드시 다시 명시하라. "
    "결론을 한 문장으로만 답하라."
)

ans_A = ask(prompt_A, reasoning_effort="low", max_completion_tokens=2000)
ans_B = ask(prompt_B, reasoning_effort="low", max_completion_tokens=2000)

compare("A) 지시대명사(그것/그 프로젝트)", ans_A, "B) 풀네임 반복 강제", ans_B)

# 정답은 '추천 시스템'(제우스). 키워드로 자동 채점.
print("\n[채점] 정답 키워드 '추천' 포함 여부")
print("  A:", keyword_hits(ans_A, ["추천"]))
print("  B:", keyword_hits(ans_B, ["추천"]))

## Tip 16 — 추론량 줄이기 (파라미터 + 자연어)

**요지**: reasoning 모델(답하기 전에 내부적으로 생각을 길게 이어가는 모델)은 간단한 질문에도 추론을 길게 돌려 `reasoning_tokens`(추론에 쓰는 토큰으로, 비용과 응답 지연으로 이어진다)를 많이 쓴다. `reasoning_effort="low/minimal"` **파라미터 하나만으로는 충분히 줄지 않을 수 있다** — 시스템/프롬프트에 자연어로도 "추론을 최소화하고 바로 답하라"를 **함께** 적어주면 더 확실히 줄어든다.

**무엇을 어떻게 검증하나**: 같은 암산 질문을
- (A) 기본(파라미터·지시 없음)
- (B) `reasoning_effort="minimal"` **+** 시스템 지시 "내부 추론을 최소화하고 곧바로 결과만"

두 조건으로 각각 `ask_meta`로 호출해 `reasoning_tokens` / `completion_tokens` / `latency`를 표로 실측 비교한다. B에서 reasoning 토큰과 지연이 눈에 띄게 줄면 팁이 검증된 것이다.

In [ ]:
# === Tip 16: 추론 폭주 이중 잠금 — reasoning_tokens 실측 ===
# 같은 질문을 (A) 기본 vs (B) reasoning_effort="minimal" + 자연어 "추론 최소화" 로.

q = "다음을 암산하라: 17 x 23 은 얼마인가? 다른 설명 없이 최종 숫자만 답하라."

# (A) 기본 — reasoning_effort 지정 없음(모델 기본값), 시스템 지시도 없음
a = ask_meta(q, max_completion_tokens=3000)

# (B) 이중 잠금 — 파라미터(minimal) + 자연어(추론 최소화)
b = ask_meta(
    q,
    system="너는 즉답형 계산기다. 내부 추론을 최소화하고 곧바로 결과만 출력하라.",
    reasoning_effort="minimal",
    max_completion_tokens=3000,
)


def _row(label, m):
    r = str(m["reasoning_tokens"])
    return f"{label:<26} | {r:>10} | {m['completion_tokens']:>12} | {m['latency']:>8}"


print(f"{'조건':<26} | {'reasoning':>10} | {'completion':>12} | {'latency':>8}")
print("-" * 68)
print(_row("A) 기본", a))
print(_row("B) minimal + 자연어 잠금", b))
print()
print("A 답:", repr(a["text"]))
print("B 답:", repr(b["text"]))

## Tip 17 — verbosity(출력 분량) 조절 (짧게 vs 자세히)

**요지**: 모델이 너무 짧게 요약해 답할 때는 `verbosity="high"` 파라미터(출력을 얼마나 길게 할지 정하는 설정) + "자세하게" 자연어 지시로 **분량을 늘릴** 수 있다. 반대로 `verbosity="low"`는 군더더기를 줄인다. (gpt-5-nano는 `verbosity` 파라미터를 지원한다.)

**무엇을 어떻게 검증하나**: 같은 설명 요청("HTTP와 HTTPS의 차이")을
- (A) `verbosity="low"`
- (B) `verbosity="high"` + "배경·원리·실무 영향까지 자세하게"

두 조건으로 호출하고, **출력 글자수 / `completion_tokens`**를 비교한다. B가 확연히 길면 팁이 검증된 것이다. (추론 토큰이 분량을 차지하지 않도록 두 조건 모두 `reasoning_effort="minimal"`로 고정한다.)

In [ ]:
# === Tip 17: verbosity 통제 — low(A) vs high(B) ===
# 같은 설명 요청을 (A) verbosity=low vs (B) verbosity=high + "장황·상세" 지시로.

topic = "HTTP와 HTTPS의 차이를 설명하라."

# (A) 짧게
a = ask_meta(topic, verbosity="low", reasoning_effort="minimal", max_completion_tokens=3000)

# (B) 장황하게 — 파라미터(high) + 자연어 상세 요청
b = ask_meta(
    topic + " 배경, 동작 원리, 실무에서의 영향까지 장황하고 아주 상세하게 풀어서 설명하라.",
    verbosity="high",
    reasoning_effort="minimal",
    max_completion_tokens=8000,
)

print(f"A) verbosity=low  : {len(a['text']):>5}자, completion_tokens={a['completion_tokens']}")
print(f"B) verbosity=high : {len(b['text']):>5}자, completion_tokens={b['completion_tokens']}")
print()
compare("A) verbosity=low", a["text"], "B) verbosity=high + 상세 요청", b["text"])

## Tip 18 — 리스트 대신 산문으로

**요지**: 최신 모델은 설명을 요청하면 습관적으로 `1. 2. 3.` 번호나 하이픈 불릿으로 **개조식** 나열을 하는 경향이 있다. 보고서 본문·에세이처럼 **줄글(산문)**을 원하면 "불릿·번호·하이픈 금지, 자연스러운 산문 문단으로만"이라고 명시해야 한다.

**무엇을 어떻게 검증하나**: 같은 주제("마이크로서비스 아키텍처의 장점")를
- (A) 기본 (모델이 알아서 → 대개 개조식으로)
- (B) "불릿·번호·하이픈 금지, 산문 문단으로만"

두 조건으로 호출한 뒤, **줄 맨 앞의 리스트 마커(`-`, `*`, `•`, `1.`) 개수**를 세어 정량 비교한다. B에서 마커 수가 0에 가깝게 떨어지면 팁이 검증된 것이다.

In [ ]:
# === Tip 18: 개조식 집착 역해제 — 기본(A) vs 산문 강제(B) ===
# 줄 시작의 리스트 마커(-, *, •, 1.) 개수를 세어 형식 차이를 정량화.

import re

topic = "마이크로서비스 아키텍처의 장점을 설명하라."

# (A) 기본 — 모델이 알아서 (대개 개조식/불릿으로 뱉음)
a = ask(topic, verbosity="medium", reasoning_effort="minimal", max_completion_tokens=3000)

# (B) 산문 강제
b = ask(
    topic + " 단, 불릿·번호 매기기·하이픈 나열을 절대 사용하지 말고, "
    "자연스럽게 이어지는 산문 문단으로만 작성하라.",
    verbosity="medium",
    reasoning_effort="minimal",
    max_completion_tokens=3000,
)


def count_list_markers(text):
    # 줄 맨 앞의 -, *, • 또는 '1.' / '1)' 같은 번호 매기기를 리스트 마커로 카운트
    n = 0
    for line in (text or "").splitlines():
        if re.match(r"^\s*([-*•]|\d+[.)])\s+", line):
            n += 1
    return n


print(f"A) 기본       : 리스트 마커 {count_list_markers(a)}개")
print(f"B) 산문 강제  : 리스트 마커 {count_list_markers(b)}개")
print()
compare("A) 기본(개조식 허용)", a, "B) 산문 강제(리스트 금지)", b)

## 정리 — 무엇을 보면 팁이 검증되는가

- **Tip 13 (지시대명사)**: B(풀네임 반복)의 `keyword_hits` 점수(정답 "추천" 포함)가 A(지시대명사)보다 꾸준히 높으면 검증. 여러 단계를 거슬러 올라가는 질문일수록 풀네임 반복이 참조 오류를 줄인다.
- **Tip 16 (추론량 줄이기)**: 표에서 B의 `reasoning_tokens`와 `latency`가 A보다 확연히 작으면 검증. 파라미터(`minimal`)와 자연어 지시를 **함께** 걸 때 추론량이 가장 잘 줄어든다.
- **Tip 17 (verbosity)**: B(high + 상세 지시)의 글자수·`completion_tokens`가 A(low)보다 크게 많으면 검증. 분량은 `verbosity` 파라미터 + 자연어로 양쪽 방향으로 조절된다.
- **Tip 18 (리스트 해제)**: B(산문 지시)의 리스트 마커 개수가 A보다 뚜렷이 적으면(이상적으로 0) 검증. 산문을 원하면 "리스트·하이픈 금지"를 명시해야 한다.

**공통 관찰 포인트**: gpt-5-nano는 `temperature`가 1로 고정이라, 창의성·분량·형식은 오직 **프롬프트 텍스트와 `reasoning_effort` / `verbosity` 파라미터**로만 조절된다. 또 reasoning 모델이라 `max_completion_tokens`가 너무 작으면 추론이 예산을 다 써서 본문이 빈 문자열이 될 수 있으니 넉넉히 준다. 소형 모델 특성상 한 번의 실행이 절대적이지 않으므로, 각 셀을 여러 번 Run 해 경향을 확인하라.